# НМ: CNN „пукнатина / без пукнатина“ за дрон инспектор на сгради
**Тема 2 – Дрон инспектор на сгради** · част по „Невронни мрежи“

**Задача на мрежата.** Дронът снима стена; снимката се нарязва на плочки 227 × 227. Мрежата казва за всяка плочка вероятността $\hat y = P(\text{пукнатина})$. От вероятностите и избран праг получаваме **карта 0/1** – входът за АСД (зони) и ЕМП (маршрут).

**Сравняваме два модела:**
1. **Собствен CNN** – обучен от нулата.
2. **MobileNetV2 с трансферно обучение** – предварително обучен върху ImageNet, дообучен за нашата задача.

**Най-важна метрика – пълнота (recall)** за класа „пукнатина“: пропусната пукнатина (FN) е по-скъпа от излишна проверка (FP). Затова прагът се избира по валидационното множество така, че пълнотата да е ≥ 99 %.

> ⚙️ Среда: Google Colab → *Runtime → Change runtime type → T4 GPU*. Цялата тетрадка се изпълнява за ~25–40 минути на T4.

In [ ]:
# Настройка: в Colab клонираме хранилището, локално ползваме папката src/
import os, sys
REPO_URL = "https://github.com/<потребител>/nm-cnn-dron-<екип>.git"  # ← сменете с адреса на вашето хранилище
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/repo"):
        !git clone -q $REPO_URL /content/repo
    os.chdir("/content/repo")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Работна папка:", os.getcwd())

In [ ]:
import os, time, random, json, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, classification_report, precision_score, recall_score,
                             f1_score, precision_recall_curve, roc_auc_score, ConfusionMatrixDisplay)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or "няма (ще е бавно)")

# ---- Настройки ----
SMOKE = os.environ.get("SMOKE_TEST") == "1"   # бърза проверка на кода със синтетични данни
IMG_OWN = 128          # вход на собствения CNN (227 -> 128: 3 пъти по-малко изчисления, пукнатините остават видими)
IMG_TL = 160           # вход на MobileNetV2 (поддържани размери: 96, 128, 160, 192, 224)
BATCH = 64
EPOCHS_OWN = 30        # максимум – ранното спиране обикновено спира по-рано
EPOCHS_HEAD, EPOCHS_FT = 5, 5
SUBSET = None          # напр. 10000 за по-бързи опити; None = всички 40 000 снимки
TARGET_RECALL = 0.99   # изискване за пълнота при избора на праг
if SMOKE:
    IMG_OWN, IMG_TL, BATCH, EPOCHS_OWN, EPOCHS_HEAD, EPOCHS_FT = 64, 96, 16, 2, 1, 1
Path("results").mkdir(exist_ok=True); Path("models").mkdir(exist_ok=True)

## 1. Данни
**Concrete Crack Images for Classification** – Özgenel, Ç. F. (2019), Mendeley Data, V2, DOI [10.17632/5y9wdsg2zt.2](https://doi.org/10.17632/5y9wdsg2zt.2).
* 40 000 RGB снимки 227 × 227, 2 класа по 20 000: `Positive` (пукнатина) и `Negative` (без пукнатина).
* Получени от 458 снимки 4032 × 3024 на бетонни стени в кампуса на ODTÜ (Анкара) – т.е. наборът **вече е плочки от по-големи снимки** – точно нашият сценарий.
* Лиценз: посочен на страницата на набора (CC BY 4.0) – цитираме авторите (виж „Източници“ в края и `data/README.md`).

**Как да се изтеглят** (изберете един начин):
1. **Официален източник (Mendeley):** изтеглете архива от страницата → качете го в Google Drive → задайте `ARCHIVE` по-долу.
2. **Огледало в Kaggle** (`arunrk7/surface-crack-detection` – същите 40 000 снимки) – клетката опитва автоматично с `kagglehub`.
3. Ако вече имате разархивирани папки `Positive/` и `Negative/` – задайте `DATA_ROOT`.

In [ ]:
DATA_ROOT = Path("/content/data" if IN_COLAB else "data/raw")
ARCHIVE = None   # напр. "/content/drive/MyDrive/Concrete Crack Images for Classification.rar"

def find_class_dirs(root):
    """Търси папка, в която има Positive/ и Negative/ (на произволна дълбочина)."""
    for p in Path(root).rglob("Positive"):
        if p.is_dir() and (p.parent / "Negative").is_dir():
            return p.parent
    return None

def make_synthetic(root, n=120, size=227):
    """САМО за SMOKE_TEST: тъмни линии върху шумен фон – проверява кода, не е за отчитане."""
    from PIL import Image, ImageDraw
    rng = np.random.default_rng(0)
    for cls in ("Positive", "Negative"):
        (root / cls).mkdir(parents=True, exist_ok=True)
        for i in range(n):
            img = Image.fromarray((rng.normal(170, 18, (size, size, 3))).clip(0, 255).astype("uint8"))
            if cls == "Positive":
                d = ImageDraw.Draw(img)
                pts = [tuple(rng.integers(0, size, 2)) for _ in range(4)]
                d.line(pts, fill=(40, 40, 40), width=int(rng.integers(2, 6)))
            img.save(root / cls / f"{i:05d}.jpg")

base = find_class_dirs(DATA_ROOT) if DATA_ROOT.exists() else None
if base is None and SMOKE:
    make_synthetic(DATA_ROOT / "synthetic"); base = find_class_dirs(DATA_ROOT)
if base is None and ARCHIVE:
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    if str(ARCHIVE).endswith(".rar"):
        !unrar x -inul -o+ "{ARCHIVE}" "{DATA_ROOT}/"
    else:
        shutil.unpack_archive(ARCHIVE, DATA_ROOT)
    base = find_class_dirs(DATA_ROOT)
if base is None:
    try:
        import kagglehub
        base = find_class_dirs(kagglehub.dataset_download("arunrk7/surface-crack-detection"))
    except Exception as e:
        print("kagglehub не успя:", e)
assert base is not None, "Данните не са намерени – вижте инструкциите горе и data/README.md"

pos = sorted(str(p) for p in (base / "Positive").glob("*.jpg"))
neg = sorted(str(p) for p in (base / "Negative").glob("*.jpg"))
print(f"Папка: {base}\nPositive (пукнатина): {len(pos)} | Negative (без): {len(neg)}")

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for row, (files, name) in enumerate([(pos, "пукнатина"), (neg, "без пукнатина")]):
    for ax, f in zip(axes[row], random.sample(files, 6)):
        ax.imshow(plt.imread(f)); ax.set_title(name, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.savefig("results/nm_examples.png", dpi=110); plt.show()

### Разделяне: обучаващо 70 % / валидационно 15 % / тестово 15 %
Стратифицирано (еднакъв дял на класовете във всяко множество), с фиксиран seed. Списъците се записват в `results/split.json`, за да е разделянето възпроизводимо.

> **Правило за честна оценка:** изборът на модел, праг и настройки става **само по валидационното** множество. Тестовото се използва **веднъж, накрая**.

In [ ]:
files = np.array(pos + neg)
labels = np.array([1] * len(pos) + [0] * len(neg))
if SUBSET:
    idx = np.random.default_rng(SEED).choice(len(files), SUBSET, replace=False)
    files, labels = files[idx], labels[idx]
X_tr, X_tmp, y_tr, y_tmp = train_test_split(files, labels, test_size=0.30, stratify=labels, random_state=SEED)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)
for n, y in [("обучаващо", y_tr), ("валидационно", y_val), ("тестово", y_te)]:
    print(f"{n:<13} {len(y):>6} снимки, дял пукнатини {y.mean():.3f}")
json.dump({"train": X_tr.tolist(), "val": X_val.tolist(), "test": X_te.tolist()},
          open("results/split.json", "w"))

### Входен конвейер (tf.data)
* декодиране на JPEG → смяна на размера → кеширане в паметта като `uint8` (по-бързи епохи);
* **аугментация само за обучаващото множество**: хоризонтално/вертикално огледало, завъртане на 90°, яркост и контраст. Пукнатината остава пукнатина при завъртане – така мрежата не „заучава“ посоката.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load(path, label, size):
    img = tf.image.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.image.resize(img, (size, size), antialias=True)
    return tf.cast(tf.clip_by_value(img, 0, 255), tf.uint8), label

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.rot90(img, tf.random.uniform([], 0, 4, dtype=tf.int32))
    img = tf.cast(img, tf.float32)
    img = tf.image.random_brightness(img, 25.0)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    return tf.clip_by_value(img, 0, 255), label

def make_ds(X, y, size, training):
    ds = tf.data.Dataset.from_tensor_slices((X, y.astype("float32")))
    ds = ds.map(lambda p, l: load(p, l, size), num_parallel_calls=AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(4096, seed=SEED).map(augment, num_parallel_calls=AUTOTUNE)
    else:
        ds = ds.map(lambda i, l: (tf.cast(i, tf.float32), l))
    return ds.batch(BATCH).prefetch(AUTOTUNE)

train_own, val_own, test_own = (make_ds(X, y, IMG_OWN, t) for X, y, t in
                                [(X_tr, y_tr, True), (X_val, y_val, False), (X_te, y_te, False)])

## 2. Модел 1: собствен CNN

### Архитектура
```
вход 128×128×3 → Rescaling(1/255)
[Conv 3×3, 32] → BN → ReLU → [Conv 3×3, 32] → BN → ReLU → MaxPool 2×2      → 64×64×32
[Conv 3×3, 64] → BN → ReLU → [Conv 3×3, 64] → BN → ReLU → MaxPool 2×2      → 32×32×64
[Conv 3×3,128] → BN → ReLU → [Conv 3×3,128] → BN → ReLU → MaxPool 2×2      → 16×16×128
[Conv 3×3,256] → BN → ReLU → GlobalAveragePooling                         → 256
Dropout 0.4 → Dense 64, ReLU, L2 → Dropout 0.3 → Dense 1, sigmoid          → ŷ ∈ (0, 1)
```

### Формули
* **Конволюция** (филтър $K$ с размер $k\times k$, канали $c$): $\; z_{i,j}^{(f)} = b^{(f)} + \sum_{u=0}^{k-1}\sum_{v=0}^{k-1}\sum_{c} K^{(f)}_{u,v,c}\, x_{i+u,\,j+v,\,c}$
* **ReLU:** $g(z) = \max(0, z)$ · **сигмоида** (изход): $\hat y = \sigma(z) = \dfrac{1}{1+e^{-z}}$
* **Пакетна нормализация:** $\hat z = \dfrac{z-\mu_B}{\sqrt{\sigma_B^2+\varepsilon}},\; z' = \gamma \hat z + \beta$ – стабилизира и ускорява обучението.
* **Функция на загубата** – бинарна кросентропия + L2:
$$J = -\frac{1}{m}\sum_{i=1}^{m}\Big[y_i\log\hat y_i + (1-y_i)\log(1-\hat y_i)\Big] + \lambda\sum_l \lVert W_l\rVert_2^2$$
* **Dropout:** при обучение всеки неврон се изключва с вероятност p – мрежата не разчита на отделни неврони (по-малко преобучение).
* **Оптимизатор Adam:** $m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t,\; v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2,\; w \leftarrow w - \alpha \dfrac{\hat m_t}{\sqrt{\hat v_t}+\varepsilon}$, с $\alpha = 10^{-3}, \beta_1 = 0.9, \beta_2 = 0.999$.
* **Ранно спиране:** спираме, когато загубата на валидационното не се подобрява 5 епохи, и връщаме най-добрите тегла.

### Защо тази архитектура
* Пукнатината е **локален признак** (тъмна тънка линия) → малки филтри 3×3, наредени по два, виждат 5×5 със същия брой параметри като едно 5×5, но с още една нелинейност.
* **GlobalAveragePooling** вместо Flatten → ~100 пъти по-малко параметри в плътния слой → по-малко преобучение; няма значение *къде* в плочката е пукнатината.
* BN + Dropout + L2 + аугментация + ранно спиране – справочници 1–4.

In [ ]:
L2 = regularizers.l2(1e-4)

def conv_bn(x, f):
    x = layers.Conv2D(f, 3, padding="same", use_bias=False, kernel_regularizer=L2)(x)
    x = layers.BatchNormalization()(x)
    return layers.ReLU()(x)

def build_own_cnn(size):
    inp = keras.Input((size, size, 3))
    x = layers.Rescaling(1 / 255.0)(inp)
    for f in (32, 64, 128):
        x = conv_bn(x, f); x = conv_bn(x, f); x = layers.MaxPooling2D()(x)
    x = conv_bn(x, 256)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation="relu", kernel_regularizer=L2)(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inp, out, name="sobstven_cnn")

METRICS = [keras.metrics.BinaryAccuracy(name="acc"), keras.metrics.Precision(name="precision"),
           keras.metrics.Recall(name="recall"), keras.metrics.AUC(name="auc")]
own = build_own_cnn(IMG_OWN)
own.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=METRICS)
own.summary()

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6),
]
t0 = time.time()
hist_own = own.fit(train_own, validation_data=val_own, epochs=EPOCHS_OWN, callbacks=callbacks, verbose=2)
time_own = time.time() - t0
print(f"Обучение: {time_own/60:.1f} мин, {len(hist_own.history['loss'])} епохи")

In [ ]:
def plot_history(hist, title, path):
    h = hist.history if hasattr(hist, "history") else hist
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
    for a, k, name in zip(ax, ["loss", "acc", "recall"], ["загуба", "точност", "пълнота"]):
        a.plot(h[k], "o-", label="обучаващо"); a.plot(h["val_" + k], "o-", label="валидационно")
        a.set_title(f"{title}: {name}"); a.set_xlabel("епоха"); a.legend()
    plt.tight_layout(); plt.savefig(path, dpi=110); plt.show()

plot_history(hist_own, "Собствен CNN", "results/nm_curves_own.png")

**Как четем кривите:** ако валидационната загуба започне да расте, докато обучаващата пада → преобучение (ранното спиране връща теглата от най-добрата епоха). Ако двете остават високи → недообучение (по-голям модел / повече епохи).

## 3. Модел 2: MobileNetV2 с трансферно обучение
**Идея:** първите слоеве на мрежа, обучена върху 1.2 млн. снимки (ImageNet), вече разпознават ръбове, линии и текстури – точно това, което ни трябва за пукнатини. Използваме ги наготово и обучаваме само нова „глава“.

**MobileNetV2** (Sandler et al., 2018) е създадена за мобилни устройства – подходяща за **бордовия компютър на дрон**:
* **дълбочинно разделими конволюции:** обикновена конволюция $k\times k$ с $C_{in}\to C_{out}$ канала струва $k^2 C_{in} C_{out}$ умножения на пиксел; разделена на дълбочинна ($k^2 C_{in}$) + точкова 1×1 ($C_{in}C_{out}$) – ~8–9 пъти по-малко при k = 3;
* **обърнати остатъчни блокове** с линейно „гърло“ (inverted residuals, linear bottleneck).

**Два етапа:**
1. **Замразена основа**, обучаваме само главата (GAP → Dropout → Dense 1), Adam $10^{-3}$.
2. **Фино настройване:** размразяваме последните ~30 слоя, много малка скорост $10^{-5}$ – иначе „разваляме“ научените признаци. BN слоевете остават в режим на извод (`training=False`).

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

train_tl, val_tl, test_tl = (make_ds(X, y, IMG_TL, t) for X, y, t in
                             [(X_tr, y_tr, True), (X_val, y_val, False), (X_te, y_te, False)])

base_model = MobileNetV2(input_shape=(IMG_TL, IMG_TL, 3), include_top=False, weights="imagenet")
base_model.trainable = False
inp = keras.Input((IMG_TL, IMG_TL, 3))
x = preprocess_input(inp)                   # [0, 255] -> [-1, 1], както при обучението върху ImageNet
x = base_model(x, training=False)           # BN в режим на извод
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(1, activation="sigmoid")(x)
tl = keras.Model(inp, out, name="mobilenetv2_transfer")
tl.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=METRICS)
print("Обучаеми параметри (етап 1):", sum(int(np.prod(w.shape)) for w in tl.trainable_weights))

t0 = time.time()
h1 = tl.fit(train_tl, validation_data=val_tl, epochs=EPOCHS_HEAD, verbose=2)

# Етап 2: фино настройване на последните 30 слоя
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False
tl.compile(optimizer=keras.optimizers.Adam(1e-5), loss="binary_crossentropy", metrics=METRICS)
print("Обучаеми параметри (етап 2):", sum(int(np.prod(w.shape)) for w in tl.trainable_weights))
h2 = tl.fit(train_tl, validation_data=val_tl, epochs=EPOCHS_HEAD + EPOCHS_FT, initial_epoch=len(h1.history["loss"]),
            callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
            verbose=2)
time_tl = time.time() - t0
hist_tl = {k: h1.history[k] + h2.history[k] for k in h1.history}
plot_history(hist_tl, "MobileNetV2", "results/nm_curves_tl.png")

## 4. Избор на праг по валидационното множество
По подразбиране $\hat y \ge 0.5$ → „пукнатина“. Но цената на грешките **не е еднаква**:
* **FN** (пропусната пукнатина) → повредата остава незабелязана → риск за сградата;
* **FP** (излишна проверка) → дронът облита още една плочка → няколко секунди батерия.

Затова избираме **най-високия праг, при който пълнотата на валидационното множество е ≥ 99 %** – така максимизираме прецизността при гарантирана пълнота.

In [ ]:
def predict(model, ds):
    return model.predict(ds, verbose=0).ravel()

def choose_threshold(y_true, p, target_recall=TARGET_RECALL):
    prec, rec, thr = precision_recall_curve(y_true, p)
    ok = np.where(rec[:-1] >= target_recall)[0]          # rec[i] съответства на thr[i]
    i = ok[np.argmax(thr[ok])] if len(ok) else 0
    return float(thr[i]), prec, rec, thr

p_val = {"Собствен CNN": predict(own, val_own), "MobileNetV2": predict(tl, val_tl)}
thresholds = {}
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, p in p_val.items():
    t, prec, rec, thr = choose_threshold(y_val, p)
    thresholds[name] = t
    ax[0].plot(rec, prec, label=name)
    ax[1].plot(thr, rec[:-1], label=f"{name}: пълнота"); ax[1].plot(thr, prec[:-1], "--", label=f"{name}: прецизност")
    ax[1].axvline(t, color="gray", lw=0.8)
    print(f"{name}: праг = {t:.3f} | вал. пълнота {recall_score(y_val, p >= t):.4f}, "
          f"прецизност {precision_score(y_val, p >= t, zero_division=0):.4f}")
ax[0].set_xlabel("пълнота"); ax[0].set_ylabel("прецизност"); ax[0].set_title("Крива прецизност–пълнота (валидационно)"); ax[0].legend()
ax[1].set_xlabel("праг"); ax[1].set_title("Пълнота и прецизност спрямо прага"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig("results/nm_threshold.png", dpi=110); plt.show()

## 5. Оценка върху тестовото множество (единствен път)
* **Прецизност** $P = \dfrac{TP}{TP+FP}$ – от обявените пукнатини колко са истински;
* **Пълнота** $R = \dfrac{TP}{TP+FN}$ – от истинските пукнатини колко сме намерили (**най-важна**);
* $F_1 = \dfrac{2PR}{P+R}$ – хармонично средно;
* **матрица на объркванията** – редове: истински клас, колони: предсказан.

In [ ]:
rows = []
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
test_sets = {"Собствен CNN": (own, test_own), "MobileNetV2": (tl, test_tl)}
p_test = {}
for ax, (name, (model, ds)) in zip(axes, test_sets.items()):
    t0 = time.time(); p = predict(model, ds); dt = (time.time() - t0) / len(y_te) * 1000
    p_test[name] = p
    pred = (p >= thresholds[name]).astype(int)
    cm = confusion_matrix(y_te, pred)
    ConfusionMatrixDisplay(cm, display_labels=["без", "пукнатина"]).plot(ax=ax, colorbar=False, values_format="d")
    ax.set_title(f"{name} (праг {thresholds[name]:.2f})")
    rows.append({"модел": name, "праг": round(thresholds[name], 3),
                 "точност": (pred == y_te).mean(), "прецизност": precision_score(y_te, pred),
                 "пълнота": recall_score(y_te, pred), "F1": f1_score(y_te, pred),
                 "ROC-AUC": roc_auc_score(y_te, p), "FN": int(cm[1, 0]), "FP": int(cm[0, 1]),
                 "параметри": model.count_params(), "ms/снимка": dt})
    print(f"\n=== {name} ===\n" + classification_report(y_te, pred, target_names=["без", "пукнатина"], digits=4))
plt.tight_layout(); plt.savefig("results/nm_confusion.png", dpi=110); plt.show()

import pandas as pd
table = pd.DataFrame(rows).set_index("модел")
table.to_csv("results/nm_metrics.csv")
table.style.format({"точност": "{:.4f}", "прецизност": "{:.4f}", "пълнота": "{:.4f}", "F1": "{:.4f}",
                    "ROC-AUC": "{:.4f}", "параметри": "{:,}", "ms/снимка": "{:.2f}"})

### Грешките: кои пукнатини мрежата пропуска (FN) и кои чисти плочки обявява за пукнатина (FP)

In [ ]:
best_name = max(rows, key=lambda r: (r["пълнота"], r["F1"]))["модел"]
p, t = p_test[best_name], thresholds[best_name]
fn = np.where((y_te == 1) & (p < t))[0]; fp = np.where((y_te == 0) & (p >= t))[0]
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for row, (idx, name) in enumerate([(fn, "FN – пропусната"), (fp, "FP – излишна")]):
    for k, ax in enumerate(axes[row]):
        ax.axis("off")
        if k < len(idx):
            i = idx[k]; ax.imshow(plt.imread(X_te[i])); ax.set_title(f"{name}\nŷ = {p[i]:.2f}", fontsize=8)
plt.suptitle(f"Грешки на {best_name} върху тестовото множество"); plt.tight_layout()
plt.savefig("results/nm_errors.png", dpi=110); plt.show()
print(f"{best_name}: FN = {len(fn)}, FP = {len(fp)} от {len(y_te)} тестови снимки")

## 6. Сравнение: силни и слаби страни
*Попълнете с вашите числа от таблицата горе.*

| | Собствен CNN | MobileNetV2 (трансферно) |
|---|---|---|
| **Силни** | малко параметри; пълен контрол и разбиране на архитектурата; не зависи от външни тегла | готови признаци от ImageNet → висока точност с малко епохи; оптимизирана за мобилни устройства |
| **Слаби** | обучава се по-дълго от нулата; при малко данни се преобучава по-лесно | повече параметри; ImageNet снимките са различни от бетон; изисква точно определена предобработка |
| **Кога** | ако бордовият компютър е много слаб | по подразбиране за дрона |

**Ограничения на набора:** всички снимки са от един кампус, при сходно осветление и един тип бетон; плочките са вече изрязани и центрирани. Върху реални снимки от дрон (размазване, сенки, мръсотия, фуги между панели, графити) точността ще е по-ниска – нужен е допълнителен тест с реални снимки.

## 7. Към конвейера: от снимка на стена до карта 0/1
Нямаме снимки на цели стени с вярна карта, затова **сглобяваме стена** 20 × 30 плочки от **тестовите** снимки: където случайната маска е 1, слагаме плочка с пукнатина, иначе – без. Така знаем верния отговор и можем да измерим грешката на целия конвейер.

Стъпки: стена (4540 × 6810 px) → нарязване на плочки 227 × 227 → модел → вероятности → праг → **карта 0/1** → `results/crack_map.npy` → АСД (зони) → ЕМП (маршрут).

In [ ]:
from src.tiling import slice_into_tiles, probs_to_map, random_crack_mask, assemble_wall, map_errors
from src.pipeline import run_pipeline, report, InspectionConfig
from src.viz import plot_pipeline

ROWS, COLS = 20, 30
true_map = random_crack_mask(ROWS, COLS, n_cracks=9, max_len=7, seed=2)
te_pos = [X_te[i] for i in np.where(y_te == 1)[0][:400]]
te_neg = [X_te[i] for i in np.where(y_te == 0)[0][:400]]
from PIL import Image
read = lambda f: np.asarray(Image.open(f).convert("RGB"))
wall = assemble_wall(true_map, [read(f) for f in te_pos], [read(f) for f in te_neg], seed=0)
print("Снимка на стената:", wall.shape)

tiles, shape = slice_into_tiles(wall, tile=227)          # (600, 227, 227, 3), (20, 30)
model, size = (tl, IMG_TL) if best_name == "MobileNetV2" else (own, IMG_OWN)
x = tf.image.resize(tiles.astype("float32"), (size, size), antialias=True)
probs = model.predict(x, batch_size=BATCH, verbose=0).ravel()
crack_map = probs_to_map(probs, shape, thresholds[best_name])
np.save("results/crack_map.npy", crack_map)
print("Грешки по плочки:", map_errors(crack_map, true_map))

fig, ax = plt.subplots(1, 4, figsize=(18, 3.6))
ax[0].imshow(wall); ax[0].set_title("сглобена стена"); ax[0].axis("off")
ax[1].imshow(probs.reshape(shape), cmap="magma", vmin=0, vmax=1); ax[1].set_title("вероятност ŷ")
ax[2].imshow(crack_map, cmap="Reds"); ax[2].set_title(f"карта 0/1 (праг {thresholds[best_name]:.2f})")
ax[3].imshow(crack_map - true_map, cmap="bwr", vmin=-1, vmax=1); ax[3].set_title("грешки: синьо = FN, червено = FP")
plt.tight_layout(); plt.savefig("results/nm_wall_map.png", dpi=110); plt.show()

In [ ]:
out = run_pipeline(crack_map, InspectionConfig(battery=60))
print(report(out))
plot_pipeline(crack_map, out, "results/pipeline_from_cnn.png"); plt.show()

### (По избор) Собствена снимка на стена
Качете снимка (напр. от телефон, ~1–2 m от стената) и пуснете клетката. Плочките трябва да са в приблизително същия мащаб като в набора (пукнатината да заема видима част от 227 × 227).

In [ ]:
UPLOAD = False   # True -> качване на снимка в Colab
if UPLOAD and IN_COLAB:
    from google.colab import files as gfiles
    from PIL import Image
    name = next(iter(gfiles.upload()))
    img = np.array(Image.open(name).convert("RGB"))
    tiles, shape = slice_into_tiles(img, 227)
    x = tf.image.resize(tiles.astype("float32"), (size, size), antialias=True)
    my_map = probs_to_map(model.predict(x, verbose=0).ravel(), shape, thresholds[best_name])
    fig, ax = plt.subplots(1, 2, figsize=(12, 4)); ax[0].imshow(img); ax[1].imshow(my_map, cmap="Reds"); plt.show()

In [ ]:
own.save("models/sobstven_cnn.keras"); tl.save("models/mobilenetv2_transfer.keras")
json.dump({k: float(v) for k, v in thresholds.items()}, open("results/thresholds.json", "w"), ensure_ascii=False)
print("Записани модели в models/ (не се качват в GitHub – големи са) и прагове в results/thresholds.json")

## 8. Изводи (попълнете с вашите резултати)
1. Кой модел има по-висока пълнота при прага, избран по валидационното? Колко FN остават от ~3000 тестови пукнатини?
2. Колко прецизност „платихме“ за пълнота ≥ 99 % (сравнете с праг 0.5)?
3. Какво показват грешките (FN/FP) – тънки пукнатини, сенки, грапава повърхност?
4. Какво ще се промени при реални снимки от дрон и как бихме го подобрили (повече разнообразни данни, сегментация вместо класификация, по-малки плочки с припокриване)?

## Източници
1. Özgenel, Ç. F. (2019). *Concrete Crack Images for Classification*, Mendeley Data, V2. DOI: 10.17632/5y9wdsg2zt.2
2. Özgenel, Ç. F., Gönenç Sorguç, A. (2018). *Performance Comparison of Pretrained Convolutional Neural Networks on Crack Detection in Buildings*. ISARC 2018, Berlin.
3. Zhang, L., Yang, F., Zhang, Y. D., Zhu, Y. J. (2016). *Road crack detection using deep convolutional neural network*. IEEE ICIP.
4. Sandler, M., Howard, A., Zhu, M., Zhmoginov, A., Chen, L.-C. (2018). *MobileNetV2: Inverted Residuals and Linear Bottlenecks*. CVPR.
5. Kingma, D. P., Ba, J. (2015). *Adam: A Method for Stochastic Optimization*. ICLR.
6. Ioffe, S., Szegedy, C. (2015). *Batch Normalization*. ICML. · Srivastava, N. et al. (2014). *Dropout*. JMLR.
7. Документация на TensorFlow/Keras: keras.io/api/applications/mobilenet/